# 04. Creating an Agent with the Built-in Web Search Tool

Unlike `03_invoke_agent.py`, this script does **not** reuse the `IT-HelpDesk-Agent` from earlier in the
section — it creates a **brand-new** agent, `"web-search-lab-agent"`, from scratch using
`client.agents.create_version(...)`, the same call as `02_prompt_agent.py`. The only new idea here is the
`tools=[WebSearchTool()]` argument: this attaches Azure AI Foundry's **built-in, server-executed** web search
tool to the agent, so it can ground answers in current information without you writing any retrieval code.

**Difficulty:** Beginner


## Prerequisites

**pip3 packages:** `azure-ai-projects`, `azure-identity`, `python-dotenv` — already in root `requirements.txt`.
`WebSearchTool` is exported from `azure.ai.projects.models`, same package, no extra install.

**Azure resource(s) required:**
- A Foundry project with a deployed chat model.
- The **web search** built-in tool must be enabled/available for your Foundry project/region (built-in tools
  can have their own prerequisites/connections configured in the Foundry portal — check your project's Tools
  settings if this call fails).

**Environment variables** (root `.env`):
- `AZURE_AI_PROJECT_ENDPOINT`
- `AZURE_AI_MODEL_DEPLOYMENT`


## What You'll Learn

- How to attach a **built-in tool** (`WebSearchTool`) to a `PromptAgentDefinition`, as opposed to a custom
  `FunctionTool`
- The difference between tools the **service executes for you** (web search, code interpreter, file search)
  and tools **you must execute yourself** (`FunctionTool`, seen in `05_IT_HelpDesk_agent.py`)
- That `create_version` with a new `agent_name` creates an entirely separate agent, distinct from re-versioning an existing one


### Setup and creating the agent with `WebSearchTool`

Same client construction pattern as before, refactored to read from environment variables. The new element is
`tools=[WebSearchTool()]` inside the `PromptAgentDefinition` — this is a zero-configuration, built-in tool: you
don't implement any search logic, Foundry's service performs the search and feeds results back into the
model's context automatically during generation.

- **💡 Exam tip:** AI-102 groups Foundry Agent Service tools into roughly two families: **built-in/hosted
  tools** (Web Search, Code Interpreter, File Search / Azure AI Search grounding) that execute *inside* the
  service, and **custom tools** (`FunctionTool`) that the *client* must execute and report back. Knowing which
  family a tool belongs to tells you who is responsible for running it.
- **🔄 Alternatives:** `FileSearchTool` (grounds answers in an uploaded document / vector store — see
  `11_azure_ai_foundry/07_knowledge_rag`) or a custom `FunctionTool` calling a real search API would be
  alternatives to `WebSearchTool` depending on whether you need open-web grounding vs. private-document
  grounding vs. a specific external API.


In [ ]:
import os

from dotenv import load_dotenv
from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import PromptAgentDefinition, WebSearchTool
from azure.identity import DefaultAzureCredential

load_dotenv()

PROJECT_ENDPOINT = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
DEPLOYMENT_NAME = os.getenv("AZURE_AI_MODEL_DEPLOYMENT", "gpt-5.4")
WEB_SEARCH_AGENT_NAME = os.getenv("AZURE_AI_WEB_SEARCH_AGENT_NAME", "web-search-lab-agent")

client = AIProjectClient(
    endpoint=PROJECT_ENDPOINT,
    credential=DefaultAzureCredential(),
)

agent = client.agents.create_version(
    agent_name=WEB_SEARCH_AGENT_NAME,
    definition=PromptAgentDefinition(
        model=DEPLOYMENT_NAME,
        instructions=(
            "You are a helpful assistant. Use web search to answer questions that require current information."
        ),
        tools=[WebSearchTool()],
    )
)

print("Agent created:")
print(f"  ID      : {agent.id}")
print(f"  Name    : {agent.name}")
print(f"  Version : {agent.version}")


## Summary

This notebook creates a new, independent Foundry agent — `web-search-lab-agent` — whose `instructions` tell it
to use web search for time-sensitive questions, and whose `tools` list attaches the built-in `WebSearchTool`.
Nothing here touches the `IT-HelpDesk-Agent` from earlier scripts; this is a separate agent identity entirely.
The printed `id`/`name`/`version` confirm the new agent was registered in your Foundry project.


## Try It Yourself

1. **Easy:** After creating the agent, invoke it the same way `03_invoke_agent.py` invokes
   `IT-HelpDesk-Agent` (swap the `agent_reference` name), asking a question that requires current information
   (e.g. "What's a notable AI news story from this week?").
2. **Intermediate:** Combine `WebSearchTool()` with a `FunctionTool` in the same `tools=[...]` list and observe
   (via the response's tool-call trace, if your SDK version exposes it) which tool the model chooses for
   different questions.
3. **Advanced:** Research (Foundry docs) whether `WebSearchTool` supports configuration such as result count or
   domain restrictions, and, if so, add that configuration here.
